# Tutorial 3: group statistical comparison

This notebook introduces the object that compares groups directly: `GroupComparison`.

The goal is to:

- define the two biological groups to compare
- run the statistical test for reactions and genes
- define significance thresholds
- inspect the main result tables
- visualize the distributions with histograms and scatter plots

## Before you run it

The notebook is self-contained and repeats the setup from the previous tutorials. It keeps the same Th17 Compass dataset so that only the analysis layer changes.

In [13]:
import os
import sys
import tempfile
import warnings
from pathlib import Path

from bokeh.io import output_notebook


os.environ.setdefault(
    "MPLCONFIGDIR",
    str(Path(tempfile.gettempdir()) / "metabis-matplotlib"),
)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", message="IProgress not found.*")


def find_metabis_repo_root_for_import(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]

    for candidate in [*direct_candidates, *nested_candidates]:
        if (candidate / "src" / "metabis").exists():
            return candidate

    raise FileNotFoundError("Could not locate the packaged Metabis repository root.")


repo_root_for_import = find_metabis_repo_root_for_import(Path.cwd().resolve())
src_path = repo_root_for_import / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from metabis import (
    DataManager,
    DataManagerMode,
    ExperimentAnalysis,
    ExpressionData,
    GroupComparison,
    MetabolicModel,
    ReactionData,
    plot_statistical_histograms,
    plot_statistical_scatterplots,
)

output_notebook()

Loading BokehJS ...

In [14]:
def find_metabis_repo_root(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]

    for candidate in [*direct_candidates, *nested_candidates]:
        if (
            (candidate / "pyproject.toml").exists()
            and (candidate / "demo").exists()
            and (candidate / "src" / "metabis").exists()
        ):
            return candidate

    raise FileNotFoundError("Could not locate the packaged Metabis repository root.")


def resolve_example_data_dir(repo_root: Path) -> Path:
    data_dir = repo_root / "data"

    if (
        (data_dir / "Th17_Compass_reactions.csv").exists()
        and (data_dir / "Th17_Compass_expression.csv").exists()
    ):
        return data_dir

    raise FileNotFoundError(
        "Could not find the bundled Th17 Compass example files in metabis/data."
    )


repo_root = find_metabis_repo_root(Path.cwd().resolve())
data_dir = resolve_example_data_dir(repo_root)
reaction_data_path = data_dir / "Th17_Compass_reactions.csv"
expression_data_path = data_dir / "Th17_Compass_expression.csv"
analysis_group_1 = "Th17n"
analysis_group_2 = "Th17p"

reaction_loading_options = {
    "data_mode": "dual",
    "reaction_name_format": "rxnRecon3DID",
    "csv_separator": "\t",
    "samples_in": "columns",
    "check_old_identifiers": True,
}

expression_loading_options = {
    "gene_format": "geneSymbols",
    "normalization_options": ["CPM"],
    "data_format": "csv",
    "csv_separator": "\t",
    "samples_in": "columns",
}

model = MetabolicModel(use_cache=True)
alignment_file_path = model.get_alignment_file_path("RECON2")
model.apply_alignment_corrections(alignment_file_path)

reaction_data, _ = ReactionData.create_with_validation(
    str(reaction_data_path),
    metabolic_model=model,
    **reaction_loading_options,
)
expression_data, _ = ExpressionData.create_with_validation(
    str(expression_data_path),
    gene_list=model.get_genes(),
    metabolic_model=model,
    **expression_loading_options,
)

dm = DataManager(model, reaction_data, expression_data, None)
dm.load_groups(criteria="sample_prefix")

print(f"Groups available for comparison: {dm.get_group_ids()}")

Groups available for comparison: ['Th17p', 'Th17n']


## 1. Initialize experiment analysis, then set the active analysis group and run the comparison

Before continuing, initialize `ExperimentAnalysis` and set the active analysis group explicitly.

This is the setup that keeps the comparison plots direction-aware. The statistical plots and hover content reuse reaction-direction information stored in `ExperimentAnalysis`, and that state is interpreted with respect to the active analysis group. In this tutorial we use `analysis_group_1`, which is `Th17n`.

In [15]:
analysis = ExperimentAnalysis(dm)
dm.update_experiment_analysis(analysis)
dm.set_analysis_group(analysis_group_1)

group_comparison = GroupComparison(
    analysis_group_1,
    analysis_group_2,
    test="wilcoxon",
    correction="fdr_bh",
)

group_comparison.run_test(dm, for_genes=False)
group_comparison.run_test(dm, for_genes=True)

reaction_results = group_comparison.get_test_results(for_genes=False)
gene_results = group_comparison.get_test_results(for_genes=True)

print(f"Active analysis group: {dm.get_active_group()}")
print(f"Reaction result rows: {reaction_results.shape[0]}")
print(f"Gene result rows: {gene_results.shape[0]}")

Active analysis group: Th17n
Reaction result rows: 8462
Gene result rows: 1532


## 2. Define significant elements

The comparison object can keep all tested elements, but in practice it is useful to register a threshold configuration so that the plotting helpers know which elements are currently considered significant.

In [16]:
analysis_settings = {
    "pval_threshold": 0.10,
    "cohens_d_threshold": 0.0,
    "log2_fc_threshold": 0.0,
    "mean_difference_threshold": 0.0,
    "keep_reversed": True,
}

significant_reactions, reaction_threshold_info = group_comparison.define_significant_elements(
    pval_threshold=analysis_settings["pval_threshold"],
    cohens_d_threshold=analysis_settings["cohens_d_threshold"],
    log2_fc_threshold=analysis_settings["log2_fc_threshold"],
    mean_difference_threshold=analysis_settings["mean_difference_threshold"],
    keep_reversed=analysis_settings["keep_reversed"],
    for_genes=False,
 )
significant_genes, gene_threshold_info = group_comparison.define_significant_elements(
    pval_threshold=analysis_settings["pval_threshold"],
    cohens_d_threshold=analysis_settings["cohens_d_threshold"],
    log2_fc_threshold=analysis_settings["log2_fc_threshold"],
    mean_difference_threshold=analysis_settings["mean_difference_threshold"],
    keep_reversed=analysis_settings["keep_reversed"],
    for_genes=True,
 )

dm.update_group_comparison(group_comparison)
dm.set_mode(DataManagerMode.GROUP_COMPARISON)

print(f"Current mode: {dm.get_mode()}")
print(f"Significant reactions: {len(significant_reactions)}")
print(f"Significant genes: {len(significant_genes)}")
print(
    "Thresholds applied: "
    f"adjusted p <= {analysis_settings['pval_threshold']}, "
    f"|Cohen's d| >= {analysis_settings['cohens_d_threshold']}, "
    f"|log2 FC| >= {analysis_settings['log2_fc_threshold']}, "
    f"|mean difference| >= {analysis_settings['mean_difference_threshold']}"
 )

Current mode: DataManagerMode.GROUP_COMPARISON
Significant reactions: 2363
Significant genes: 430
Thresholds applied: adjusted p <= 0.1, |Cohen's d| >= 0.0, |log2 FC| >= 0.0, |mean difference| >= 0.0


## 3. Inspect the result tables

The reaction and gene tables are regular pandas DataFrames, so you can sort, filter, and export them with standard pandas workflows.

In [17]:
reaction_results.sort_values("adjusted_pval")[
    ["adjusted_pval", "cohens_d", "log2_fc", "mean_difference", "subsystems"]
].head(10)

,adjusted_pval,cohens_d,log2_fc,mean_difference,subsystems
MAR07142,2.105739e-48,-2.190686,-0.009850,-0.067192,Nicotinate and nicotinamide metabolism
MAR08788,2.105739e-48,-2.190686,-0.009850,-0.067192,Nicotinate and nicotinamide metabolism
MAR11427_back,5.177308e-47,-1.214606,-0.010173,-0.069402,Exchange/demand reactions
MAR11426_back,5.177308e-47,-1.247594,-0.004325,-0.029046,Exchange/demand reactions
MAR04423_back,5.177308e-47,-1.204431,-0.011835,-0.082041,Arginine and proline metabolism
MAR07152,3.418524e-20,-1.226395,-0.003173,-0.008991,Transport reactions
MAR00396_back,3.418524e-20,-1.226395,-0.003173,-0.008991,Transport reactions
MAR00397_back,3.418524e-20,-1.226395,-0.003173,-0.008991,Fatty acid activation (cytosolic)
MAR09375,4.500004e-20,-1.445785,-0.003929,-0.022131,Exchange/demand reactions
MAR07905,7.802026e-20,-1.382746,-0.008515,-0.050830,Transport reactions


In [18]:
gene_results.sort_values("adjusted_pval")[
    ["adjusted_pval", "cohens_d", "log2_fc", "mean_difference", "subsystems"]
].head(10)

,adjusted_pval,cohens_d,log2_fc,mean_difference,subsystems
ENSG00000165272,1.297342e-47,-1.223751,-12.064740,-848.570434,Transport reactions
ENSG00000086062,2.329101e-39,-1.737588,-2.363062,-336.714347,"N-glycan metabolism, Sphingolipid metabolism, ..."
ENSG00000167996,7.043177e-33,-1.712829,-0.913256,-4969.968101,Porphyrin metabolism
ENSG00000130707,2.918158e-30,-1.406225,-2.779339,-606.480465,"Alanine, aspartate and glutamate metabolism"
ENSG00000146648,4.409315e-29,0.945389,7.077030,398.405259,Transport reactions
ENSG00000164039,2.926946e-27,-1.090105,-3.138613,-89.152586,"Bile acid biosynthesis, Propanoate metabolism,..."
ENSG00000130066,1.798665e-26,1.515905,1.183999,4306.671901,Arginine and proline metabolism
ENSG00000133313,2.696666e-24,-1.129903,-2.300435,-477.099825,"Histidine metabolism, Isolated, Transport reac..."
ENSG00000140459,2.696666e-24,-0.941360,-5.264640,-258.552702,Steroid metabolism
ENSG00000172543,9.126163e-24,-0.818072,-40.265264,-132.145423,Protein degradation


## 4. Histogram view of the comparison

`plot_statistical_histograms()` summarizes the global distribution of the chosen metrics and overlays the currently active significance thresholds.

In [19]:
plot_statistical_histograms(
    dm,
    parameters_to_plot=["pval", "cohens_d", "log2_fc"],
    return_plot=False,
    show_thresholds=True,
    for_genes=False,
)

## 5. Scatter views of the same metrics

`plot_statistical_scatterplots()` is useful when you want to see how p-values, effect sizes, and fold changes relate to each other for the same set of reactions.

In [20]:
plot_statistical_scatterplots(
    dm,
    parameters_to_plot=["pval", "cohens_d", "log2_fc"],
    return_plot=False,
    show_thresholds=True,
    for_genes=False,
)

## Takeaways

At this point the `DataManager` holds a `GroupComparison` object, which is the state needed for subsystem ranking and metabolic graph coloring.

The next notebook builds on this comparison and moves into pathway-level interpretation with subsystem plots and metabolic graphs.